# Lesson 6: Structuring Projects with Classes

## Learning Objectives

By the end of this lesson, you will be able to:
- Understand the Single Responsibility Principle (SRP) and apply it to class design.
- Design a small system of interacting classes to solve a problem.
- Explain the concept of Composition ("has-a" relationship) as an alternative to Inheritance.
- Structure a simple data science project by separating concerns into different classes.

## Why This Topic Matters

So far, we've learned the mechanics of OOP: creating classes, inheriting from them, and using polymorphism and magic methods. Now, we need to address the bigger question: **how do we use these tools to organize a real project?**

In a typical data science script, it's easy to end up with a long, tangled file that mixes data loading, cleaning, feature engineering, model training, and evaluation all in one place. This 'spaghetti code' is hard to read, difficult to debug, and nearly impossible to reuse.

OOP provides a powerful way to fight this complexity. By designing a system of small, focused classes that work together, we can create projects that are modular, testable, and much easier to maintain and scale. This lesson is about moving from writing *code* to designing *systems*.

## The Problem: A Monolithic Script

Let's look at a typical script for a simple machine learning task. We want to load some data, clean it, train a model, and make a prediction.

*(This is a simplified example. Imagine this script being 500 lines long with more complex logic.)*

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# --- A 'Spaghetti Code' Example ---

# 1. Load Data
print("Loading data...")
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

# 2. Preprocess Data
print("Preprocessing data...")
# Fill missing 'Age' values
df['Age'].fillna(df['Age'].median(), inplace=True)
# Drop rows with missing 'Embarked'
df.dropna(subset=['Embarked'], inplace=True)
# Convert 'Sex' to numeric
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
# Select features and target
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch']
target = 'Survived'
X = df[features]
y = df[target]

# 3. Train Model
print("Training model...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression()
model.fit(X_train, y_train)

# 4. Evaluate Model
print("Evaluating model...")
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {accuracy:.4f}")

This works, but it has problems:
- **Hard to Read:** You have to read the whole script to understand the workflow.
- **Hard to Reuse:** What if you want to use the same preprocessing logic on a different dataset? You'd have to copy-paste code.
- **Hard to Test:** How do you test just the data cleaning part without running the whole script?
- **Hard to Change:** What if you want to try a different model? Or a different way of filling missing values? You have to carefully find and change code in the middle of the script.

## The Solution: The Single Responsibility Principle (SRP)

The **Single Responsibility Principle** is a core concept in software design. It states:

> **A class should have only one reason to change.**

This means each class should have one, and only one, job. 

Looking at our script, we can identify several distinct jobs:
- Loading data
- Preprocessing data
- Training a model
- Evaluating a model
- Managing the overall workflow

Let's refactor our script into a system of classes, where each class has a single responsibility.

### Step 1: Design the Classes

1.  `DataLoader`: Its only job is to load data from a source. It doesn't know or care about cleaning or modeling.
2.  `DataPreprocessor`: Its only job is to clean and transform the data. It receives raw data and outputs clean data.
3.  `ModelTrainer`: Its only job is to train a machine learning model. It receives clean data and outputs a trained model.
4.  `Pipeline`: Its job is to manage the overall workflow. It will *own* instances of the other classes and call them in the correct order. This is the conductor of our orchestra.

### Step 2: Implement the Classes

Now let's write the code for each class. Notice how clean and focused each one is.

In [ ]:
# --- Class Implementations ---

class DataLoader:
    """Responsibility: Load data from a given path."""
    def __init__(self, path):
        self.path = path
        
    def load(self):
        print(f"Loading data from {self.path}")
        return pd.read_csv(self.path)

class DataPreprocessor:
    """Responsibility: Clean and prepare the Titanic dataset."""
    def __init__(self, features, target):
        self.features = features
        self.target = target
        
    def process(self, df):
        print("Processing data...")
        df_processed = df.copy()
        df_processed['Age'].fillna(df_processed['Age'].median(), inplace=True)
        df_processed.dropna(subset=['Embarked'], inplace=True)
        df_processed['Sex'] = df_processed['Sex'].map({'male': 0, 'female': 1})
        
        X = df_processed[self.features]
        y = df_processed[self.target]
        return X, y

class ModelTrainer:
    """Responsibility: Train a model and evaluate it."""
    def __init__(self, model, random_state=42):
        self.model = model
        self.random_state = random_state
        
    def train_and_evaluate(self, X, y):
        print(f"Training {self.model.__class__.__name__}...")
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=self.random_state
        )
        self.model.fit(X_train, y_train)
        predictions = self.model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        print(f"Model Accuracy: {accuracy:.4f}")
        return self.model

## Composition: The "Has-A" Relationship

Notice that our `Pipeline` class won't *be* a `DataLoader` or *be* a `ModelTrainer`. Instead, it will *have* a `DataLoader` and *have* a `ModelTrainer` as attributes. This is called **Composition**.

- **Inheritance (Is-A):** A `CsvLoader` **is a** `DataLoader`. Use this for specialization.
- **Composition (Has-A):** A `Pipeline` **has a** `DataLoader`. Use this to build complex objects from smaller, independent parts.

A common design principle is **"Favor composition over inheritance."** It often leads to more flexible and less coupled designs. Our `Pipeline` is a perfect example of this. It coordinates the other objects, but it doesn't inherit their specific logic.

### Step 3: The `Pipeline` Class (The Conductor)

The `Pipeline` class will take the other objects as inputs in its `__init__` method. This is a powerful pattern called **Dependency Injection**. It means the `Pipeline` is not responsible for creating its own dependencies; they are given to it. This makes it incredibly flexible and testable.

In [ ]:
class Pipeline:
    """Responsibility: Manage the entire ML workflow."""
    def __init__(self, data_loader, preprocessor, model_trainer):
        # The pipeline HAS-A data_loader, preprocessor, and model_trainer
        self.data_loader = data_loader
        self.preprocessor = preprocessor
        self.model_trainer = model_trainer
        
    def run(self):
        """Execute the steps in order."""
        print("--- Starting Pipeline ---")
        # 1. Use the data_loader to load data
        raw_data = self.data_loader.load()
        
        # 2. Use the preprocessor to clean the data
        X, y = self.preprocessor.process(raw_data)
        
        # 3. Use the model_trainer to train and evaluate
        trained_model = self.model_trainer.train_and_evaluate(X, y)
        
        print("--- Pipeline Finished ---")
        return trained_model

### Step 4: Putting It All Together

Now, our main script becomes incredibly simple and readable. It's just about configuring and running the pipeline.

In [ ]:
# --- Configuration ---
DATA_PATH = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
FEATURES = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch']
TARGET = 'Survived'

# --- Assembly ---

# 1. Create the component objects
loader = DataLoader(path=DATA_PATH)
preprocessor = DataPreprocessor(features=FEATURES, target=TARGET)
trainer = ModelTrainer(model=LogisticRegression())

# 2. Inject dependencies into the pipeline
titanic_pipeline = Pipeline(
    data_loader=loader,
    preprocessor=preprocessor,
    model_trainer=trainer
)

# --- Execution ---
final_model = titanic_pipeline.run()

Look at the benefits:

- **Readable:** The `Pipeline.run()` method reads like a high-level summary of the project.
- **Reusable:** We can easily create a new `DataLoader` for a different file format and plug it into the pipeline without changing anything else.
- **Testable:** We can test `DataPreprocessor` in isolation by giving it a sample DataFrame and checking its output.
- **Flexible:** Want to try a different model? Just pass a different model object to the `ModelTrainer`!

For example, let's try a `RandomForestClassifier`:

In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("\n--- Running pipeline with a different model ---")

# All we change is the model we pass to the trainer!
rf_trainer = ModelTrainer(model=RandomForestClassifier(random_state=42))

rf_pipeline = Pipeline(
    data_loader=loader,
    preprocessor=preprocessor,
    model_trainer=rf_trainer
)

final_rf_model = rf_pipeline.run()

## Recap

- **Single Responsibility Principle (SRP):** The most important principle for organizing code. Each class should do one thing and do it well.
- **Composition ("Has-A"):** A powerful way to build complex systems. A `Pipeline` *has a* `DataLoader`. This is often more flexible than inheritance.
- **Dependency Injection:** Passing dependencies (like the `loader` and `trainer`) into a class's `__init__` method. This decouples your classes and makes your system highly configurable and testable.
- By combining these ideas, you can transform a messy script into a clean, professional, and maintainable data science project.

## Suggested Next Lesson

We have now seen how to design a system of interacting classes. The final step is to put all our knowledge together—from basic classes to inheritance, polymorphism, and project structure—to build a complete, end-to-end machine learning pipeline as a final case study. This will solidify all the concepts we've learned and show how they apply in a realistic scenario.